In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

--2026-11-02 00:53:11--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv’

car_fuel_efficiency 100%[===================>] 620.29K  --.-KB/s    in 0.04s   

2026-11-02 00:53:11 (13.8 MB/s) - ‘car_fuel_efficiency_2026.csv’ saved [635180/635180]



In [5]:
df = pd.read_csv('/kaggle/working/car_fuel_efficiency_2026.csv')
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [ ]:
def transform(df):
    dicts = df.to_dict(orient='records')

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

cat_cols = df.select_dtypes(include='object').columns
num_cols = df.select_dtypes(exclude='object').columns
df[cat_cols] = df[cat_cols].fillna('NA')
df[num_cols] = df[num_cols].fillna(0.0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

# Remove the target from the feature dataframes
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

dv = DictVectorizer(sparse=True)
train_dicts = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts);

In [16]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=1)

In [18]:
val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

y_pred = dt.predict(X_val)

In [20]:
# Q1
from sklearn.tree import export_text

features = dv.get_feature_names_out()
print(export_text(dt, feature_names=list(features)))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [22]:
# Q2
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
root_mean_squared_error(y_pred, y_val)

1.837054054730018

In [23]:
# Q3
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_pred, y_val)
    print(f"n_estimators = {n} -> rmse = {rmse}")

n_estimators = 10 -> rmse = 1.837054054730018
n_estimators = 50 -> rmse = 1.7715666428333987
n_estimators = 100 -> rmse = 1.7681362324492989
n_estimators = 150 -> rmse = 1.7688040324216567


In [24]:
# Q4
for depth in [10, 15, 20, 25]:
    sum_rmse = 0
    count = 0
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_pred, y_val)
        sum_rmse += rmse
        count += 1
        print(f"n_estimators = {n} -> rmse = {rmse}")
    
    print(f"depth = {depth}, mean RMSE = {sum_rmse / count}")

n_estimators = 10 -> rmse = 1.783692896135791
n_estimators = 50 -> rmse = 1.7485674334671282
n_estimators = 100 -> rmse = 1.744839046550913
n_estimators = 150 -> rmse = 1.7445197692662424
depth = 10, mean RMSE = 1.7554047863550186
n_estimators = 10 -> rmse = 1.8298920576324103
n_estimators = 50 -> rmse = 1.7711408474614405
n_estimators = 100 -> rmse = 1.7678741797717459
n_estimators = 150 -> rmse = 1.7669036906209203
depth = 15, mean RMSE = 1.7839526938716292
n_estimators = 10 -> rmse = 1.8410312717201658
n_estimators = 50 -> rmse = 1.7731091634666318
n_estimators = 100 -> rmse = 1.7691518334361584
n_estimators = 150 -> rmse = 1.769676005654852
depth = 20, mean RMSE = 1.788242068569452
n_estimators = 10 -> rmse = 1.8367754186318759
n_estimators = 50 -> rmse = 1.7721078631501397
n_estimators = 100 -> rmse = 1.768402916621684
n_estimators = 150 -> rmse = 1.7679832571153926
depth = 25, mean RMSE = 1.786317363879773


In [41]:
# Q5
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=dv.get_feature_names_out())
candidates = ["vehicle_weight", "horsepower", "acceleration", "engine_displacement"]
importances[candidates].sort_values(ascending=False)

vehicle_weight         0.203879
horsepower             0.078017
engine_displacement    0.061005
acceleration           0.055201
dtype: float64

In [42]:
!pip install xgboost

In [44]:
# Q6
import xgboost as xgb

features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

watchlist = [(dtrain, "train"), (dval, "val")]

for eta in [0.3, 0.1]:
    xgb_params = {
        "eta": eta,
        "max_depth": 6,
        "min_child_weight": 1,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "nthread": 8,
        "seed": 1,
        "verbosity": 1,
    }
    evals_result = {}
    xgb.train(xgb_params, dtrain, num_boost_round=100,
              evals=watchlist, evals_result=evals_result, verbose_eval=False)

    val_rmse = evals_result["val"]["rmse"]
    print(f"eta={eta}: final val RMSE {val_rmse[-1]:.4f}, "
          f"best {min(val_rmse):.4f} at round {val_rmse.index(min(val_rmse))}")
    

eta=0.3: final val RMSE 1.8265, best 1.7320 at round 10
eta=0.1: final val RMSE 1.7248, best 1.7122 at round 39
